# Notebook 7 — Confirmation on CMU-MOSI (same code, same settings, no tuning)

**Purpose.** Everything found on MOSEI (Notebooks 4–6) was exploratory, because the MOSEI test set was examined many times.
This notebook repeats the chain on a dataset we have **not** looked at, with **identical** code and settings:
1. train M1–M4 (v2 code) and the dedicated audio/video models U1–U3 (Notebook 4), seeds 42/43/44;
2. extract M4's classifier input for all 7 modality combinations and train the Notebook 6 heads (H1–H4), plus one new
   control, **H1F** = H1 but with the epoch chosen on the full condition only (as M4 does) — this separates
   "re-training the head on frozen features" from "choosing the epoch on all combinations";
3. the M1+U3 late-fusion baseline.

**Pre-registered checks** (primary metric: 3-class macro-F1, same as MOSEI; Acc-2 on non-neutral clips reported as a
robustness check because MOSI has only 30 neutral test clips). Intervals: seed+video bootstrap 95 %.
| Check | Comparison (V and A+V) | MOSEI result |
|---|---|---|
| C1 gap exists | dedicated − M4 own | +2.41 / +1.98 pp |
| C2 head re-fit recovers it | H2 − M4 own | +1.72 / +1.38 pp |
| C3 no conditional boundaries | H4 − H2 | +0.88 / +0.63 pp (n.s.) |
| C4 full condition unchanged | H2 − M4 own on T+A+V | +0.16 pp |
| C5 (new) selection vs re-training | H1 − H1F | not measured on MOSEI |

Reading for C1/C2: **confirmed** = same sign as MOSEI and interval excludes 0; **consistent but inconclusive** = same sign,
interval includes 0; **not replicated** = opposite sign. C3 confirmed if H4 − H2 ≤ 1 pp or not significant. C4 confirmed
if |H2 − M4 own| ≤ 1 pp on T+A+V.

**Caution, stated in advance:** MOSI is small (1,284 training clips; test 686 clips from 31 videos; 30 neutral test clips),
so intervals will be wide; "inconclusive" is a likely, honest outcome.

**Workload:** 21 model trainings (M1–M4, U1–U3 × 3 seeds) + 63 extraction passes + 36 head trainings
(H1, H1F, H2, H3 × 3 seeds + H4: 7 × 3). Measured in the test run before delivery (see the timing printed by each section).
Saved as it goes; after a restart, **Run all** continues. Writes only to `mosi_v7_confirmation/`.

**Runtime:** CPU + High-RAM.

In [1]:
import os, sys, time, json, random, pickle, platform, hashlib, itertools
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import sklearn
from joblib import Parallel, delayed
DEVICE = "cpu"
torch.backends.cudnn.deterministic = True

Mounted at /content/drive


## 1. Settings — identical to the MOSEI runs (copied from the v2 checkpoints' recorded settings)

In [2]:
BASE_OUT = "/content/drive/MyDrive/emofar_results"
DATA_PATH = "/content/drive/MyDrive/datasets/CMU-MOSI/Processed/aligned_50.pkl"
if TEST_MODE:
    BASE_OUT = os.environ["EMOFAR_OUT"]; DATA_PATH = os.environ["EMOFAR_MOSI"]
V2_DIR = f"{BASE_OUT}/mosei_v2_cpu"                       # read only: settings are copied from here
OUT_DIR = f"{BASE_OUT}/mosi_v7_confirmation"
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True); os.makedirs(f"{OUT_DIR}/reps", exist_ok=True); os.makedirs(f"{OUT_DIR}/heads", exist_ok=True)
mosei_cfg = json.load(open(f"{V2_DIR}/checkpoints/M4_seed42_log.json"))["train_config"]
D_MODEL, EPOCHS, PATIENCE, BATCH_SIZE, LR, MOD_DROPOUT = (mosei_cfg[k] for k in ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT"])
SEEDS = [42, 43, 44]
THREADS = 4 if not TEST_MODE else int(os.environ.get("EMOFAR_THREADS", 2)); torch.set_num_threads(THREADS)
N_JOBS = max(1, min(THREADS, os.cpu_count() or 1))
N_BOOT = 10000 if not TEST_MODE else 2000
MARGIN_PP = 1.0
ENV = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__,
       "torch_threads": torch.get_num_threads(), "n_jobs": N_JOBS, "os": platform.platform(),
       "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())}
CONFIG = dict(D_MODEL=D_MODEL, EPOCHS=EPOCHS, PATIENCE=PATIENCE, BATCH_SIZE=BATCH_SIZE, LR=LR, MOD_DROPOUT=MOD_DROPOUT,
              DATA_PATH=DATA_PATH, SEEDS=SEEDS, threads=THREADS)
print(json.dumps(ENV, indent=1)); print("Settings (identical to MOSEI):", CONFIG)
json.dump({"environment": ENV, "config": CONFIG, "n_boot": N_BOOT}, open(f"{OUT_DIR}/run_info.json", "w"), indent=2)
RUN_DIR = OUT_DIR
LABELS = ["negative", "neutral", "positive"]
MODS = ["text", "audio", "vision"]
CONDS = ["T", "A", "V", "A+V", "T+A", "T+V", "T+A+V"]
ROW = {"T": (1, 0, 0), "A": (0, 1, 0), "V": (0, 0, 1), "A+V": (0, 1, 1), "T+A": (1, 1, 0), "T+V": (1, 0, 1), "T+A+V": (1, 1, 1)}
COND_OF_ROW = {tuple(map(float, v)): i for i, v in enumerate(ROW[c] for c in CONDS)}

{
 "python": "3.13.18",
 "torch": "2.14.1+cpu",
 "numpy": "2.4.6",
 "sklearn": "1.6.1",
 "torch_threads": 4,
 "n_jobs": 4,
 "os": "Linux-6.6.159+-x86_64-with-glibc2.39",
 "started_utc": "2026-11-24 14:25:09"
}
Settings (identical to MOSEI): {'D_MODEL': 128, 'EPOCHS': 30, 'PATIENCE': 6, 'BATCH_SIZE': 64, 'LR': 0.0005, 'MOD_DROPOUT': 0.3, 'DATA_PATH': '/content/drive/MyDrive/datasets/CMU-MOSI/Processed/aligned_50.pkl', 'SEEDS': [42, 43, 44], 'threads': 4}


## 2. Data (v2 preprocessing, unchanged)

In [3]:
t0 = time.time()
with open(DATA_PATH, "rb") as f:
    raw = pickle.load(f)
print(f"Loaded in {time.time() - t0:.0f}s")

def prep(split):
    d = raw[split]
    mask = d["text_bert"][:, 1, :].astype(bool)                        # (N, 50) real-word steps
    feats = {"text": d["text"].astype(np.float32),
             "audio": np.nan_to_num(d["audio"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0),
             "vision": np.nan_to_num(d["vision"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)}
    score = d["regression_labels"].astype(np.float32)
    y = np.where(score < 0, 0, np.where(score == 0, 1, 2))
    return {"feats": feats, "mask": mask, "y": y, "score": score, "raw_text": d["raw_text"], "id": d["id"]}

data = {s: prep(s) for s in ["train", "valid", "test"]}
del raw

for m in ["audio", "vision"]:                                           # standardise with train statistics
    tr = data["train"]["feats"][m][data["train"]["mask"]]
    mu, sd = tr.mean(0), tr.std(0) + 1e-6
    for s in data:
        x = (data[s]["feats"][m] - mu) / sd
        data[s]["feats"][m] = np.clip(x, -10, 10) * data[s]["mask"][..., None]

DIMS = {m: data["train"]["feats"][m].shape[-1] for m in MODS}
print("Feature sizes:", DIMS)
for s in data:
    counts = np.bincount(data[s]["y"], minlength=3)
    print(f"{s:5s}: {len(data[s]['y']):6,} clips | " + " / ".join(f"{LABELS[i]} {counts[i]:,}" for i in range(3)))
np.save(f"{RUN_DIR}/test_labels.npy", data["test"]["y"]); np.save(f"{RUN_DIR}/valid_labels.npy", data["valid"]["y"])
np.save(f"{RUN_DIR}/test_scores.npy", data["test"]["score"])
pd.Series(data["test"]["id"]).to_csv(f"{RUN_DIR}/test_ids.csv", index=False, header=["id"])

Loaded in 5s
Feature sizes: {'text': 768, 'audio': 5, 'vision': 20}
train:  1,284 clips | negative 552 / neutral 53 / positive 679
valid:    229 clips | negative 92 / neutral 13 / positive 124
test :    686 clips | negative 379 / neutral 30 / positive 277


## 3–4. Models and training functions (v2, unchanged) + the Notebook 4 pooled MLP

In [4]:
def masked_mean(x, mask):
    m = mask.unsqueeze(-1).float()
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

class TextOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(DIMS["text"], D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(masked_mean(x["text"], mask))
        return (out, None) if return_gate else out

class PaperRMSR(nn.Module):
    """Equation-level adaptation of EMO-FAR's RMSR (Eq. 10-14)."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Linear(DIMS[m], D_MODEL) for m in MODS})          # Eq. 10
        self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})  # Eq. 11
        self.Wq, self.Wk, self.Wv = (nn.Linear(D_MODEL, D_MODEL) for _ in range(3))
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)   # Eq. 11
        gamma = q.softmax(-1)                                                             # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                     # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        return (out, gamma) if return_gate else out

class FixedFusion(nn.Module):
    """Token-level cross-attention + masked gating."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Linear(DIMS[m], D_MODEL), nn.LayerNorm(D_MODEL)) for m in MODS})
        self.xattn = nn.ModuleDict({m: nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.1)
                                    for m in ["audio", "vision"]})
        self.gate = nn.Linear(3 * D_MODEL, 3)
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(attended + p[m], mask)
        logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        return (out, w) if return_gate else out

MODELS = {"M1 Text-only": (TextOnly, 0.0),
          "M2 EMO-FAR RMSR (as written)": (PaperRMSR, 0.0),
          "M3 Fixed fusion": (FixedFusion, 0.0),
          "M4 Fixed fusion + modality dropout": (FixedFusion, MOD_DROPOUT)}
SHORT = {name: name.split()[0] for name in MODELS}                     # "M1" ... "M4"

class PooledMLP(nn.Module):          # identical to Notebook 4
    def __init__(self, mods):
        super().__init__()
        self.mods = list(mods)
        self.net = nn.Sequential(nn.Linear(sum(DIMS[m] for m in self.mods), D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(torch.cat([masked_mean(x[m], mask) for m in self.mods], -1))
        return (out, None) if return_gate else out

for name, mods in {"U1 Audio-only": ["audio"], "U2 Video-only": ["vision"], "U3 Audio+video": ["audio", "vision"]}.items():
    MODELS[name] = ((lambda mods: lambda: PooledMLP(mods))(mods), 0.0)

In [5]:
from sklearn.metrics import f1_score, accuracy_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sample_avail(n, rates, rng):
    """rates = (p_text, p_audio, p_vision): chance each modality is hidden. At least one stays."""
    avail = (rng.rand(n, 3) >= np.array(rates)).astype(np.float32)
    none_left = avail.sum(1) == 0
    avail[none_left, rng.randint(0, 3, none_left.sum())] = 1
    return avail

def to_batch(d, idx, avail):
    x = {m: torch.from_numpy(d["feats"][m][idx]).to(DEVICE) for m in MODS}
    for i, m in enumerate(MODS):
        x[m] = x[m] * torch.from_numpy(avail[:, i]).to(DEVICE)[:, None, None]
    return x, torch.from_numpy(d["mask"][idx]).to(DEVICE), torch.from_numpy(avail).to(DEVICE)

@torch.no_grad()
def predict(model, d, rates=(0, 0, 0), seed=0):
    model.eval(); rng = np.random.RandomState(seed); out = []
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, sample_avail(len(idx), rates, rng))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

def evaluate(probs, d):
    y, pred = d["y"], probs.argmax(1)
    nz = d["score"] != 0
    acc2 = ((probs[nz, 2] > probs[nz, 0]) == (d["score"][nz] > 0)).mean()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "acc2": float(acc2), "ece": ece_score(probs, y)}

def train(name, seed):
    cls, drop = MODELS[name]
    set_seed(seed)
    model = cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            loss = F.cross_entropy(model(x, mask, avail), torch.from_numpy(tr["y"][idx]).to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"  {name:38s} seed {seed}: best valid macro-F1 = {best:.4f}  (stopped after epoch {ep + 1})")
    return model, float(best), ep + 1

def save_atomic(save_fn, path):
    """Write to a temporary file first, then rename: a disconnect never leaves a half-written file."""
    tmp = path + ".tmp"
    save_fn(tmp)
    os.replace(tmp, path)

## 5. Train (or load) M1–M4 and U1–U3, 3 seeds each — saved as each finishes

In [6]:
def train_or_load(name, seed):
    tag = f"{name.split()[0]}_seed{seed}"
    ckpt, logf = f"{OUT_DIR}/checkpoints/{tag}.pt", f"{OUT_DIR}/checkpoints/{tag}_log.json"
    if os.path.exists(ckpt) and os.path.exists(logf):
        log = json.load(open(logf)); assert log["config"] == CONFIG, f"{ckpt} was trained with other settings"
        model = MODELS[name][0](); model.load_state_dict(torch.load(ckpt, map_location="cpu")); model.eval()
        return model, log, True
    t1 = time.time(); model, best, ep = train(name, seed)
    log = {"model": name, "seed": seed, "best_valid_macro_f1": best, "stopped_epoch": ep, "train_seconds": round(time.time() - t1, 1), "config": CONFIG}
    save_atomic(lambda p: torch.save(model.state_dict(), p), ckpt); save_atomic(lambda p: json.dump(log, open(p, "w"), indent=2), logf)
    for sp in ["valid", "test"]:
        np.save(f"{OUT_DIR}/checkpoints/{tag}_{sp}_probs.npy", predict(model, data[sp]))
    return model, log, False

ORDER = ["M1 Text-only", "M2 EMO-FAR RMSR (as written)", "M3 Fixed fusion", "M4 Fixed fusion + modality dropout",
         "U1 Audio-only", "U2 Video-only", "U3 Audio+video"]
MODELS_T, rows = {}, []
t0 = time.time(); n_total = len(ORDER) * len(SEEDS); k = 0
for name in ORDER:
    for s in SEEDS:
        model, log, loaded = train_or_load(name, s); MODELS_T[(name, s)] = model; k += 1
        r = evaluate(np.load(f"{OUT_DIR}/checkpoints/{name.split()[0]}_seed{s}_test_probs.npy"), data["test"])
        r.update(model=name.split()[0], seed=s, stopped_epoch=log["stopped_epoch"], train_seconds=log["train_seconds"]); rows.append(r)
        status = "loaded" if loaded else "trained in %.0fs" % log["train_seconds"]
        print(f"  [{k}/{n_total}] {name:38s} seed {s}: {status} "
              f"| test macro-F1 {r['macro_f1']:.4f}  ({time.time() - t0:.0f}s elapsed)", flush=True)
base = pd.DataFrame(rows); base.to_csv(f"{OUT_DIR}/models_per_seed.csv", index=False)
print(base.groupby("model", sort=False)[["accuracy", "macro_f1", "acc2", "ece"]].agg(["mean", "std"]).round(4).to_string())

  M1 Text-only                           seed 42: best valid macro-F1 = 0.5326  (stopped after epoch 12)
  [1/21] M1 Text-only                           seed 42: trained in 9s | test macro-F1 0.5279  (10s elapsed)
  M1 Text-only                           seed 43: best valid macro-F1 = 0.5488  (stopped after epoch 24)
  [2/21] M1 Text-only                           seed 43: trained in 4s | test macro-F1 0.5880  (15s elapsed)
  M1 Text-only                           seed 44: best valid macro-F1 = 0.5461  (stopped after epoch 19)
  [3/21] M1 Text-only                           seed 44: trained in 3s | test macro-F1 0.5727  (18s elapsed)
  M2 EMO-FAR RMSR (as written)           seed 42: best valid macro-F1 = 0.5311  (stopped after epoch 10)
  [4/21] M2 EMO-FAR RMSR (as written)           seed 42: trained in 3s | test macro-F1 0.5094  (21s elapsed)
  M2 EMO-FAR RMSR (as written)           seed 43: best valid macro-F1 = 0.5442  (stopped after epoch 11)
  [5/21] M2 EMO-FAR RMSR (as written)  

## 6. Bootstrap helpers (as in Notebooks 4–6)

In [7]:
def video_of(ids):
    return np.array([str(i).split("$_$")[0] for i in ids])

def f1_from_counts(cm):
    cm = cm.reshape(cm.shape[:-1] + (3, 3)); tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp; den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0).mean(-1)

y_test, y_val, y_tr = data["test"]["y"], data["valid"]["y"], data["train"]["y"]
_, vidx_test = np.unique(video_of(data["test"]["id"]), return_inverse=True); G_TEST = vidx_test.max() + 1
_r = np.random.RandomState(2026)
W_TEST = np.stack([np.bincount(_r.randint(0, G_TEST, G_TEST), minlength=G_TEST) for _ in range(N_BOOT)]).astype(np.float32)
SEED_DRAWS = np.random.RandomState(99).randint(0, len(SEEDS), (N_BOOT, len(SEEDS)))
mf1 = lambda y, p: f1_score(y, p, average="macro")

def video_counts(pred):
    c = np.zeros((G_TEST, 9)); np.add.at(c, (vidx_test, y_test * 3 + pred), 1); return c

def _p(b):
    return min(1.0, 2 * (min((b <= 0).sum(), (b >= 0).sum()) + 1) / (len(b) + 1))

def grouped_boot(preds_a, preds_b):
    per_seed = np.array([mf1(y_test, a) - mf1(y_test, b) for a, b in zip(preds_a, preds_b)])
    d = np.stack([f1_from_counts(W_TEST @ video_counts(a)) - f1_from_counts(W_TEST @ video_counts(b)) for a, b in zip(preds_a, preds_b)])
    sv = d[SEED_DRAWS, np.arange(N_BOOT)[:, None]].mean(1)
    return {"mean_diff_pp": 100 * per_seed.mean(), "per_seed_pp": ", ".join(f"{100*x:+.2f}" for x in per_seed),
            "seeds_positive": int((per_seed > 0).sum()), "sv_lo_pp": 100 * np.percentile(sv, 2.5), "sv_hi_pp": 100 * np.percentile(sv, 97.5),
            "sv_p": _p(sv)}

## 7. M4's classifier input for all 7 combinations (saved per seed; extraction reproduces M4's own outputs exactly)

In [8]:
@torch.no_grad()
def extract_fused(model, d, row, check=False):
    model.eval(); n = len(d["y"]); out = []
    for i in range(0, n, 512):
        idx = np.arange(i, min(i + 512, n))
        x, mask, av = to_batch(d, idx, np.tile(np.asarray(row, np.float32), (len(idx), 1)))
        p = {m: model.proj[m](x[m]) * av[:, j, None, None] for j, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            att, _ = model.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(att + p[m], mask)
        w = model.gate(torch.cat([h[m] for m in MODS], -1)).masked_fill(av == 0, float("-inf")).softmax(-1)
        fused = sum(w[:, j:j + 1] * h[m] for j, m in enumerate(MODS))
        if check:
            assert torch.equal(model.cls(fused), model(x, mask, av)), "extraction does not reproduce M4's output"
        out.append(fused.numpy())
    return np.concatenate(out)
FUSED, M4_OWN = {}, {}
t0 = time.time()
for s in SEEDS:
    f = f"{OUT_DIR}/reps/M4_seed{s}_fused.npz"; m4 = MODELS_T[("M4 Fixed fusion + modality dropout", s)]; m4.eval()
    if os.path.exists(f):
        z = np.load(f); FUSED[s] = {c: {sp: z[f"{c}|{sp}"] for sp in ["train", "valid", "test"]} for c in CONDS}
    else:
        FUSED[s] = {c: {sp: extract_fused(m4, data[sp], ROW[c], check=(sp == "test")) for sp in ["train", "valid", "test"]} for c in CONDS}
        np.savez(f, **{f"{c}|{sp}": FUSED[s][c][sp] for c in CONDS for sp in ["train", "valid", "test"]})
    with torch.no_grad():
        M4_OWN[s] = {c: m4.cls(torch.from_numpy(FUSED[s][c]["test"])).softmax(-1).numpy() for c in CONDS}
    print(f"  seed {s}: representations ready ({time.time() - t0:.0f}s elapsed)", flush=True)

  seed 42: representations ready (8s elapsed)
  seed 43: representations ready (14s elapsed)
  seed 44: representations ready (21s elapsed)


## 8. Heads H1, H1F, H2, H3, H4 (Notebook 6 code; H1F added), 4 at a time, saved as each finishes

In [9]:
class Head(nn.Module):
    """Same layers as M4's classifier: Dropout(0.2) -> Linear(128,128) -> ReLU -> Linear(128,3).
    With film=True: a per-combination scale and shift (initialised to zero) after the first Linear layer."""
    def __init__(self, film=False):
        super().__init__()
        self.drop, self.l1, self.l2 = nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.Linear(D_MODEL, 3)
        self.film = film
        if film:
            self.gamma = nn.Parameter(torch.zeros(len(CONDS), D_MODEL)); self.beta = nn.Parameter(torch.zeros(len(CONDS), D_MODEL))
    def forward(self, x, c):
        h = self.l1(self.drop(x))
        if self.film:
            h = h * (1 + self.gamma[c]) + self.beta[c]
        return self.l2(torch.relu(h))

def train_head_job(seed, variant, cond, Xtr_np, Xva_np, Xte_np):
    """variant: 'H1' (M4's exposure), 'H1F' (M4's exposure, epoch chosen on the full condition only, like M4 itself),
    'H2' (uniform), 'H3' (uniform + FiLM), 'H4' (one combination only).
    Same optimiser, learning rate, weight decay, batch size, class weights, clipping, epochs and patience as v2."""
    torch.set_num_threads(1); set_seed(seed)
    Xtr = torch.from_numpy(np.array(Xtr_np))   # private copy                                # (7, n, 128): one row per combination
    Xva = {c: torch.from_numpy(np.array(Xva_np[c])) for c in CONDS}
    head = Head(film=(variant == "H3")); opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(y_tr, minlength=3); cw = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float)
    rng = np.random.RandomState(seed); yt = torch.from_numpy(y_tr); n = len(y_tr)
    val_conds = [cond] if variant == "H4" else (["T+A+V"] if variant == "H1F" else CONDS)
    best, best_state, bad, ep_done = -1, None, 0, 0
    for ep in range(EPOCHS):
        head.train(); order = rng.permutation(n)
        for i in range(0, n, BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            if variant in ("H1", "H1F"):
                av = sample_avail(len(idx), (MOD_DROPOUT,) * 3, rng); ci = np.array([COND_OF_ROW[tuple(r)] for r in av])
            elif variant == "H4":
                ci = np.full(len(idx), CONDS.index(cond))
            else:
                ci = rng.randint(0, len(CONDS), len(idx))
            ci_t = torch.from_numpy(ci)
            loss = F.cross_entropy(head(Xtr[ci_t, torch.from_numpy(idx)], ci_t), yt[idx], weight=cw)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt.step()
        head.eval()
        with torch.no_grad():
            val = np.mean([mf1(y_val, head(Xva[c], torch.full((len(y_val),), CONDS.index(c))).argmax(1).numpy()) for c in val_conds])
        ep_done = ep + 1
        if val > best:
            best, bad = val, 0; best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    head.load_state_dict(best_state); head.eval()
    with torch.no_grad():
        test = {c: head(torch.from_numpy(np.array(Xte_np[c])), torch.full((len(y_test),), CONDS.index(c))).softmax(-1).numpy()
                for c in ([cond] if variant == "H4" else CONDS)}
    return seed, variant, cond, best, ep_done, test
XTR = {s: np.stack([FUSED[s][c]["train"] for c in CONDS]) for s in SEEDS}
JOBS = [(s, v, None) for v in ["H1", "H1F", "H2", "H3"] for s in SEEDS] + [(s, "H4", c) for c in CONDS for s in SEEDS]
tag = lambda s, v, c: f"{v}_seed{s}" + (f"_{c.replace('+', '')}" if c else "")
todo = [j for j in JOBS if not os.path.exists(f"{OUT_DIR}/heads/{tag(*j)}.json")]
print(f"Heads: {len(JOBS)} in total, {len(JOBS) - len(todo)} already saved, {len(todo)} to train on {N_JOBS} cores", flush=True)
t0 = time.time()
for k in range(0, len(todo), N_JOBS):
    chunk = todo[k:k + N_JOBS]
    jobs = [delayed(train_head_job)(s_, v_, c_, XTR[s_], {cc: FUSED[s_][cc]["valid"] for cc in CONDS},
                                    {cc: FUSED[s_][cc]["test"] for cc in CONDS}) for s_, v_, c_ in chunk]
    for s, v, c, best, ep, test in Parallel(n_jobs=N_JOBS)(jobs):
        np.savez(f"{OUT_DIR}/heads/{tag(s, v, c)}_test_probs.npz", **{cc.replace("+", "p"): p for cc, p in test.items()})
        json.dump({"seed": s, "variant": v, "condition": c, "best_valid_macro_f1": best, "stopped_epoch": ep}, open(f"{OUT_DIR}/heads/{tag(s, v, c)}.json", "w"))
    print(f"  heads saved: {len(JOBS) - len(todo) + k + len(chunk)}/{len(JOBS)}  ({time.time() - t0:.0f}s elapsed)", flush=True)

Heads: 33 in total, 0 already saved, 33 to train on 4 cores
  heads saved: 4/33  (6s elapsed)
  heads saved: 8/33  (7s elapsed)
  heads saved: 12/33  (9s elapsed)
  heads saved: 16/33  (10s elapsed)
  heads saved: 20/33  (11s elapsed)
  heads saved: 24/33  (12s elapsed)
  heads saved: 28/33  (13s elapsed)
  heads saved: 32/33  (13s elapsed)
  heads saved: 33/33  (14s elapsed)


## 9. Results for every modality combination (test macro-F1 %, mean ± SD over 3 seeds)

In [10]:
def load_head(s, v, c):
    z = np.load(f"{OUT_DIR}/heads/{tag(s, v, c)}_test_probs.npz"); return {kk.replace("p", "+"): z[kk] for kk in z.files}
PROBS = {}
for c in CONDS:
    PROBS[("M4 own", c)] = [M4_OWN[s][c] for s in SEEDS]
    for v in ["H1", "H1F", "H2", "H3"]:
        PROBS[(v, c)] = [load_head(s, v, None)[c] for s in SEEDS]
    PROBS[("H4", c)] = [load_head(s, "H4", c)[c] for s in SEEDS]
    m2 = []
    for s in SEEDS:
        mdl = MODELS_T[("M2 EMO-FAR RMSR (as written)", s)]
        with torch.no_grad():
            x, mask, av = to_batch(data["test"], np.arange(len(y_test)), np.tile(np.asarray(ROW[c], np.float32), (len(y_test), 1)))
            m2.append(mdl(x, mask, av).softmax(-1).numpy())
    PROBS[("M2 own", c)] = m2
    if "T" in c.split("+"):
        PROBS[("M1 text-only", c)] = [np.load(f"{OUT_DIR}/checkpoints/M1_seed{s}_test_probs.npy") for s in SEEDS]
for c, u in [("A", "U1"), ("V", "U2"), ("A+V", "U3")]:
    PROBS[("Dedicated model", c)] = [np.load(f"{OUT_DIR}/checkpoints/{u}_seed{s}_test_probs.npy") for s in SEEDS]
lf, lf_w = [], []
grid = np.round(np.arange(0, 0.501, 0.05), 2)
for s in SEEDS:
    m1v, m1t = np.load(f"{OUT_DIR}/checkpoints/M1_seed{s}_valid_probs.npy"), np.load(f"{OUT_DIR}/checkpoints/M1_seed{s}_test_probs.npy")
    u3v, u3t = np.load(f"{OUT_DIR}/checkpoints/U3_seed{s}_valid_probs.npy"), np.load(f"{OUT_DIR}/checkpoints/U3_seed{s}_test_probs.npy")
    w = max(grid, key=lambda w: mf1(y_val, ((1 - w) * m1v + w * u3v).argmax(1))); lf.append((1 - w) * m1t + w * u3t); lf_w.append(float(w))
PROBS[("Late fusion M1+U3", "T+A+V")] = lf
PRED = {k: [p.argmax(1) for p in v] for k, v in PROBS.items()}

nz = data["test"]["score"] != 0
acc2 = lambda p: float(((p[nz, 2] > p[nz, 0]) == (data["test"]["score"][nz] > 0)).mean())
rows = [{"method": m, "condition": c, "seed": s, "test_macro_f1": mf1(y_test, p.argmax(1)), "acc2_nonneutral": acc2(p)}
        for (m, c), ps in PROBS.items() for s, p in zip(SEEDS, ps)]
res = pd.DataFrame(rows); res.to_csv(f"{OUT_DIR}/results_per_seed.csv", index=False)
METHODS = ["M2 own", "M4 own", "H1", "H1F", "H2", "H3", "H4", "M1 text-only", "Dedicated model", "Late fusion M1+U3"]
tab = res.groupby(["method", "condition"])[["test_macro_f1", "acc2_nonneutral"]].agg(["mean", "std"])
fmt = lambda m, c, col: f"{100*tab.loc[(m, c), (col, 'mean')]:.2f} ± {100*tab.loc[(m, c), (col, 'std')]:.2f}" if (m, c) in tab.index else "—"
summary = pd.DataFrame({c: [fmt(m, c, "test_macro_f1") for m in METHODS] for c in CONDS}, index=METHODS)
summary_acc2 = pd.DataFrame({c: [fmt(m, c, "acc2_nonneutral") for m in METHODS] for c in CONDS}, index=METHODS)
summary.to_csv(f"{OUT_DIR}/results_summary_macroF1.csv"); summary_acc2.to_csv(f"{OUT_DIR}/results_summary_acc2.csv")
print("Late-fusion weights chosen on validation:", lf_w)
print("\n3-class macro-F1 (%):"); print(summary.to_string())
print("\nAcc-2 on non-neutral clips (%):"); print(summary_acc2.to_string())

Late-fusion weights chosen on validation: [0.45, 0.05, 0.45]

3-class macro-F1 (%):
                              T             A             V           A+V           T+A           T+V         T+A+V
M2 own             53.79 ± 3.61  28.20 ± 4.05  32.99 ± 0.28  33.70 ± 3.68  54.71 ± 1.81  53.70 ± 2.40  53.50 ± 3.56
M4 own             56.46 ± 0.62  23.17 ± 2.92  32.26 ± 1.96  29.03 ± 2.62  57.15 ± 0.34  56.92 ± 0.52  56.96 ± 0.61
H1                 55.66 ± 1.55  30.33 ± 3.61  30.52 ± 7.34  28.86 ± 9.04  56.45 ± 1.68  56.72 ± 0.59  55.87 ± 1.43
H1F                55.47 ± 1.48  25.47 ± 1.96  29.72 ± 6.68  28.04 ± 8.82  56.04 ± 1.35  57.29 ± 0.64  56.14 ± 1.52
H2                 57.04 ± 0.75  30.56 ± 3.54  35.85 ± 1.97  35.61 ± 2.65  56.88 ± 0.41  57.26 ± 2.37  57.52 ± 1.77
H3                 57.36 ± 1.59  33.61 ± 1.01  36.13 ± 1.80  36.59 ± 1.44  57.16 ± 0.24  57.98 ± 1.65  58.11 ± 2.05
H4                 56.78 ± 0.56  38.23 ± 2.81  36.00 ± 0.56  36.37 ± 1.53  56.92 ± 0.93  57.01 ± 1.50  5

## 10. Pre-registered checks

In [11]:
MOSEI_SIGN = {"C1": +1, "C2": +1}
checks, comp_rows = [], []
for c in ["V", "A+V"]:
    for cid, a, b in [("C1", "Dedicated model", "M4 own"), ("C2", "H2", "M4 own"), ("C3", "H4", "H2"), ("C5", "H1", "H1F")]:
        r = grouped_boot(PRED[(a, c)], PRED[(b, c)]); comp_rows.append({"check": cid, "condition": c, "comparison": f"{a} − {b}", **r})
r = grouped_boot(PRED[("H2", "T+A+V")], PRED[("M4 own", "T+A+V")]); comp_rows.append({"check": "C4", "condition": "T+A+V", "comparison": "H2 − M4 own", **r})
comp = pd.DataFrame(comp_rows); comp.to_csv(f"{OUT_DIR}/checks.csv", index=False)
def read(row):
    if row.check in ("C1", "C2"):
        same = np.sign(row.mean_diff_pp) == MOSEI_SIGN[row.check]
        return "CONFIRMED" if same and (row.sv_lo_pp > 0) else ("CONSISTENT, INCONCLUSIVE" if same else "NOT REPLICATED")
    if row.check == "C3":
        return "CONFIRMED (no conditional boundaries)" if not (row.mean_diff_pp > MARGIN_PP and row.sv_lo_pp > 0) else "NOT REPLICATED (separate heads help)"
    if row.check == "C4":
        return "CONFIRMED (full condition unchanged)" if abs(row.mean_diff_pp) <= MARGIN_PP else "NOT REPLICATED"
    return "selection effect" if row.sv_lo_pp > 0 or row.sv_hi_pp < 0 else "no clear selection effect"
comp["reading"] = comp.apply(read, axis=1)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
print(comp[["check", "condition", "comparison", "mean_diff_pp", "per_seed_pp", "sv_lo_pp", "sv_hi_pp", "reading"]].round(3).to_string(index=False))
json.dump({r.check + "_" + r.condition: r.reading for r in comp.itertuples()}, open(f"{OUT_DIR}/CONFIRMATION.json", "w"), indent=2)

check condition               comparison  mean_diff_pp         per_seed_pp  sv_lo_pp  sv_hi_pp                               reading
   C1         V Dedicated model − M4 own         2.895 +6.68, -0.72, +2.72    -1.982     8.044              CONSISTENT, INCONCLUSIVE
   C2         V              H2 − M4 own         3.590 +3.68, +3.55, +3.53     0.449     6.712                             CONFIRMED
   C3         V                  H4 − H2         0.149 +0.42, -2.41, +2.43    -4.804     4.963 CONFIRMED (no conditional boundaries)
   C5         V                 H1 − H1F         0.797 +0.00, +1.17, +1.22    -1.127     2.844             no clear selection effect
   C1       A+V Dedicated model − M4 own         4.595 +4.34, +2.14, +7.30    -0.339     9.306              CONSISTENT, INCONCLUSIVE
   C2       A+V              H2 − M4 own         6.580 +7.19, +6.74, +5.82     3.409    10.299                             CONFIRMED
   C3       A+V                  H4 − H2         0.761 +2.27, -3.77, 

## 11. Files written

In [12]:
for root, _, files in os.walk(OUT_DIR):
    if root.rstrip("/").split("/")[-1] in ("heads", "reps", "checkpoints"):
        print(f"   {os.path.relpath(root, OUT_DIR)}/  ({len(files)} files)"); continue
    for fn in sorted(files):
        print("  ", os.path.relpath(os.path.join(root, fn), OUT_DIR))

   CONFIRMATION.json
   checks.csv
   models_per_seed.csv
   results_per_seed.csv
   results_summary_acc2.csv
   results_summary_macroF1.csv
   run_info.json
   test_ids.csv
   test_labels.npy
   test_scores.npy
   valid_labels.npy
   checkpoints/  (84 files)
   reps/  (3 files)
   heads/  (66 files)
